# Metrics & Baselines

Tuần 5: Viết hàm tính điểm NWRMSLE và chạy 4 baselines trên 3 fold.

- B0: Đoán 0 hết.
- B1: Đoán bằng ngày cuối cùng đã biết.
- B2: Đoán bằng số bán cùng thứ của tuần gần nhất đã biết.
- B3: Đoán bằng trung bình 4 tuần gần nhất cùng thứ.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import timedelta
import warnings
warnings.filterwarnings('ignore')

## 1. Hàm tính NWRMSLE

In [2]:
def nwrmsle(y_true, y_pred, weights):
    """
    Tính Normalized Weighted Root Mean Squared Logarithmic Error.
    y_true, y_pred: mảng numpy chứa doanh số (chưa log).
    weights: mảng numpy chứa trọng số (1.25 cho perishable, 1.0 cho thường).
    """
    # Đảm bảo không có số âm (cắt về 0)
    y_true = np.clip(y_true, 0, None)
    y_pred = np.clip(y_pred, 0, None)
    
    log_true = np.log1p(y_true)
    log_pred = np.log1p(y_pred)
    
    squared_errors = weights * (log_true - log_pred)**2
    return np.sqrt(np.sum(squared_errors) / np.sum(weights))

# Test hàm
y_t = np.array([10, 0, 20])
y_p = np.array([10, 1, 15])
w = np.array([1.0, 1.25, 1.0])
print("Test NWRMSLE:", nwrmsle(y_t, y_p, w))

Test NWRMSLE: 0.4555686938033487


## 2. Load Dữ liệu và Tạo Lưới (Grid) validation

In [3]:
# Đọc dữ liệu train
# Vì ta chỉ cần test lại trong khoảng 3 tháng cuối để làm validation fold, ta load từ 2017-05-01
df_train = pd.read_parquet('../data/processed/train.parquet', filters=[('date', '>=', pd.Timestamp('2017-05-01'))])

# Đọc items để lấy weights
items = pd.read_csv('../data/raw/items.csv')
items['weight'] = np.where(items['perishable'] == 1, 1.25, 1.0)
item_weights = items.set_index('item_nbr')['weight'].to_dict()

# Danh sách stores và items trong scope (đã lọc ở notebook 00)
stores = df_train['store_nbr'].unique()
items_list = df_train['item_nbr'].unique()

print(f"Số cửa hàng: {len(stores)}")
print(f"Số mặt hàng: {len(items_list)}")

Số cửa hàng: 54
Số mặt hàng: 3947


## 3. Chạy các Baselines trên 3 Folds

In [4]:
folds = [
    {'name': 'Fold 1', 'val_start': '2017-07-31', 'val_end': '2017-08-15'},
    {'name': 'Fold 2', 'val_start': '2017-07-15', 'val_end': '2017-07-30'},
    {'name': 'Fold 3', 'val_start': '2017-06-29', 'val_end': '2017-07-14'}
]

results = []

for fold in folds:
    val_start = pd.to_datetime(fold['val_start'])
    val_end = pd.to_datetime(fold['val_end'])
    train_end = val_start - timedelta(days=1)
    
    print(f"\n--- {fold['name']} ---")
    print(f"Train end: {train_end.date()} | Val: {val_start.date()} to {val_end.date()}")
    
    # 1. Tạo lưới (Grid) thực tế cho tập Validation
    val_dates = pd.date_range(val_start, val_end)
    idx = pd.MultiIndex.from_product([val_dates, stores, items_list], names=['date', 'store_nbr', 'item_nbr'])
    df_val = pd.DataFrame(index=idx).reset_index()
    
    # Lấy unit_sales thực tế đắp vào Grid
    actual = df_train[(df_train['date'] >= val_start) & (df_train['date'] <= val_end)]
    df_val = df_val.merge(actual[['date', 'store_nbr', 'item_nbr', 'unit_sales']], on=['date', 'store_nbr', 'item_nbr'], how='left')
    df_val['unit_sales'] = df_val['unit_sales'].fillna(0)
    df_val['weight'] = df_val['item_nbr'].map(item_weights)
    
    y_true = df_val['unit_sales'].values
    w = df_val['weight'].values
    
    # Baseline 0: Đoán 0 hết
    y_b0 = np.zeros(len(y_true))
    score_b0 = nwrmsle(y_true, y_b0, w)
    
    # Baseline 1: Ngày cuối cùng (train_end)
    b1_data = df_train[df_train['date'] == train_end][['store_nbr', 'item_nbr', 'unit_sales']]
    b1_data.rename(columns={'unit_sales': 'pred_b1'}, inplace=True)
    df_val = df_val.merge(b1_data, on=['store_nbr', 'item_nbr'], how='left')
    y_b1 = df_val['pred_b1'].fillna(0).values
    score_b1 = nwrmsle(y_true, y_b1, w)
    
    # Baseline 2 & 3: Cùng thứ tuần trước & Trung bình 4 tuần
    # Lấy lịch sử 4 tuần trước train_end
    hist_start = train_end - timedelta(days=27) # 4 weeks = 28 days (0 to 27)
    hist_data = df_train[(df_train['date'] >= hist_start) & (df_train['date'] <= train_end)].copy()
    hist_data['dow'] = hist_data['date'].dt.dayofweek
    
    # Lọc ra tuần gần nhất cho B2
    last_week_start = train_end - timedelta(days=6)
    b2_data = hist_data[hist_data['date'] >= last_week_start].copy()
    b2_agg = b2_data.groupby(['store_nbr', 'item_nbr', 'dow'])['unit_sales'].mean().reset_index(name='pred_b2')
    
    # Lọc ra trung bình 4 tuần cho B3
    b3_agg = hist_data.groupby(['store_nbr', 'item_nbr', 'dow'])['unit_sales'].mean().reset_index(name='pred_b3')
    
    df_val['dow'] = df_val['date'].dt.dayofweek
    df_val = df_val.merge(b2_agg, on=['store_nbr', 'item_nbr', 'dow'], how='left')
    df_val = df_val.merge(b3_agg, on=['store_nbr', 'item_nbr', 'dow'], how='left')
    
    y_b2 = df_val['pred_b2'].fillna(0).values
    y_b3 = df_val['pred_b3'].fillna(0).values
    
    score_b2 = nwrmsle(y_true, y_b2, w)
    score_b3 = nwrmsle(y_true, y_b3, w)
    
    results.append({
        'Fold': fold['name'],
        'B0 (Đoán 0)': round(score_b0, 4),
        'B1 (Ngày cuối)': round(score_b1, 4),
        'B2 (Cùng thứ tuần trước)': round(score_b2, 4),
        'B3 (TB cùng thứ 4 tuần)': round(score_b3, 4)
    })
    print(f"B0: {score_b0:.4f} | B1: {score_b1:.4f} | B2: {score_b2:.4f} | B3: {score_b3:.4f}")

df_res = pd.DataFrame(results)
df_res.loc['Mean'] = df_res.mean(numeric_only=True)
df_res.at['Mean', 'Fold'] = 'Trung bình (CV)'



--- Fold 1 ---
Train end: 2017-07-30 | Val: 2017-07-31 to 2017-08-15
B0: 1.3374 | B1: 0.7610 | B2: 0.7385 | B3: 0.7030

--- Fold 2 ---
Train end: 2017-07-14 | Val: 2017-07-15 to 2017-07-30
B0: 1.3461 | B1: 0.7364 | B2: 0.7246 | B3: 0.7097

--- Fold 3 ---
Train end: 2017-06-28 | Val: 2017-06-29 to 2017-07-14
B0: 1.3599 | B1: 0.7460 | B2: 0.7213 | B3: 0.6953


## 4. Tổng kết điểm Baseline

In [5]:
display(df_res)

,Fold,B0 (Đoán 0),B1 (Ngày cuối),B2 (Cùng thứ tuần trước),B3 (TB cùng thứ 4 tuần)
0,Fold 1,1.3374,0.7610,0.738500,0.703000
1,Fold 2,1.3461,0.7364,0.724600,0.709700
2,Fold 3,1.3599,0.7460,0.721300,0.695300
Mean,Trung bình (CV),1.3478,0.7478,0.728133,0.702667
